## data ingestion & cleaning

In [35]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("theatticusproject/cuad-qa", revision="refs/convert/parquet")
train_data = dataset["train"]
test_data = dataset["test"]
print(len(train_data))
print(len(test_data))

22450
4182


In [36]:
full_data = concatenate_datasets([train_data, test_data])

In [37]:
print(len(full_data))

26632


In [38]:
full_data

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 26632
})

In [39]:
seen_titles = set()
unique_contracts = []

for row in full_data:
    if row["title"] not in seen_titles:
        seen_titles.add(row["title"])
        unique_contracts.append({
            "title": row["title"],
            "context": row["context"]
        })
print(f"Number of unique contracts: {len(unique_contracts)}")

Number of unique contracts: 510


In [40]:
import pandas as pd
df = pd.DataFrame(unique_contracts)
print(df["context"].str.len().describe())

count       510.000000
mean      52563.005882
std       55946.484865
min         645.000000
25%       16416.250000
50%       33143.000000
75%       66394.750000
max      338211.000000
Name: context, dtype: float64


In [41]:
print(df["context"].head())

0    EXHIBIT 10.6\n\n                              ...
1    Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BE...
2    Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3    REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4    Exhibit 10.23 Corporate Address Fannin South P...
Name: context, dtype: object


In [42]:
df.head()

,title,context
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...


In [43]:
for i in [0, 100]:
    print(f"Contract {i} : {unique_contracts[i]['title']}")
    print(unique_contracts[i]['context'][:800])
    print("\n\n")

Contract 0 : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its option



Contract 100 : RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT
2008 Sponsorship Agreement- Renewal Sponsor

Exhibit 

In [44]:
eval_examples = [{
    "title" : row["title"],
    "question": row["question"],
    "answers": row["answers"]["text"],
}
for row in full_data]

print(len(eval_examples))
print(eval_examples[0])

26632
{'title': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'answers': ['DISTRIBUTOR AGREEMENT']}


In [45]:
has_answer = sum(1 for ex in eval_examples if len(ex["answers"]) > 0)
no_answer = len(eval_examples) - has_answer

print(f"with answer: {has_answer} ({has_answer/len(eval_examples) * 100:.1f}%)")
print(f"without answer: {no_answer} ({no_answer/len(eval_examples) * 100:.1f}%)")

with answer: 12424 (46.7%)
without answer: 14208 (53.3%)


In [46]:
categories = set()

for ex in eval_examples:
    q = ex["question"]

    if '"' in q:
        category = q.split('"')[1]
        categories.add(category)

print(f"Number of categories: {len(categories)}")
for c in sorted(categories):
    print(f"- {c}")

Number of categories: 41
- Affiliate License-Licensee
- Affiliate License-Licensor
- Agreement Date
- Anti-Assignment
- Audit Rights
- Cap On Liability
- Change Of Control
- Competitive Restriction Exception
- Covenant Not To Sue
- Document Name
- Effective Date
- Exclusivity
- Expiration Date
- Governing Law
- Insurance
- Ip Ownership Assignment
- Irrevocable Or Perpetual License
- Joint Ip Ownership
- License Grant
- Liquidated Damages
- Minimum Commitment
- Most Favored Nation
- No-Solicit Of Customers
- No-Solicit Of Employees
- Non-Compete
- Non-Disparagement
- Non-Transferable License
- Notice Period To Terminate Renewal
- Parties
- Post-Termination Services
- Price Restrictions
- Renewal Term
- Revenue/Profit Sharing
- Rofr/Rofo/Rofn
- Source Code Escrow
- Termination For Convenience
- Third Party Beneficiary
- Uncapped Liability
- Unlimited/All-You-Can-Eat-License
- Volume Restriction
- Warranty Duration


In [47]:
lengths = df["context"].str.len()
print("smaller contract length:")
print(df.loc[lengths.nsmallest(3).index, ["title"]])
print(f"length: {lengths.nsmallest(3).values}")

print("largest contract length:")
print(df.loc[lengths.nlargest(3).index, ["title"]])
print(f"length: {lengths.nlargest(3).values}")

smaller contract length:
                                                 title
442  RMRGROUPINC_01_22_2020-EX-99.1-JOINT FILING AG...
495  PcquoteComInc_19990721_S-1A_EX-10.11_6377149_E...
501  PRECIGEN,INC_01_22_2020-EX-99.1-JOINT FILING A...
length: [645 857 972]
largest contract length:
                                                 title
358  MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-O...
157  CERES,INC_01_25_2012-EX-10.20-Collaboration Ag...
499  GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...
length: [338211 335282 300768]


In [48]:
import re

sample_text = unique_contracts[0]["context"]

multi_spaces = re.findall(r" {3,}", sample_text)
multi_newlines = re.findall(r"\n{2,}", sample_text)

print(f"Number of multi-spaces (>=3): {len(multi_spaces)}")
print(f"Number of multi-newlines (>=2): {len(multi_newlines)}")
print(f"Text length: {len(sample_text)}")

Number of multi-spaces (>=3): 900
Number of multi-newlines (>=2): 121
Text length: 54290


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

sample_titles = list(lengths.nsmallest(3).index) + list(lengths.nlargest(3).index)

for idx in sample_titles:
    text = df.loc[idx, "context"]
    n_tokens = len(tokenizer.encode(text))
    print(f"{df.loc[idx, 'title'][:50]:<50} | {len(text):>7} chars | {n_tokens:>7} tokens")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (71268 > 512). Running this sequence through the model will result in indexing errors


RMRGROUPINC_01_22_2020-EX-99.1-JOINT FILING AGREEM |     645 chars |     174 tokens
PcquoteComInc_19990721_S-1A_EX-10.11_6377149_EX-10 |     857 chars |     313 tokens
PRECIGEN,INC_01_22_2020-EX-99.1-JOINT FILING AGREE |     972 chars |     263 tokens
MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSO |  338211 chars |   71268 tokens
CERES,INC_01_25_2012-EX-10.20-Collaboration Agreem |  335282 chars |   78593 tokens
GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Franchis |  300768 chars |   61676 tokens


In [50]:
import re

def clean_contract_text(text: str) -> str:

    text = re.sub(r"[ \t]{2,}", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = "\n".join(line.strip() for line in text.split('\n'))

    return text.strip()

sample = unique_contracts[0]["context"]
cleaned = clean_contract_text(sample)

print(f"before cleaning: {len(sample)} chars")
print(f"after cleaning: {len(cleaned)} chars")
print(f"original text:\n{sample[:1000]}\n\n")
print(f"cleaned text:\n{cleaned[:1000]}")

before cleaning: 54290 chars
after cleaning: 38563 chars
original text:
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  th

In [51]:
for c in unique_contracts:
    c["clean_context"] = clean_contract_text(c["context"])

reduction = [(len(c["context"]) - len(c["clean_context"])) / len(c["context"]) * 100 for c in unique_contracts]
print(f"average reduction in length after cleaning: {sum(reduction)/len(reduction):.1f}%")

average reduction in length after cleaning: 3.6%


In [52]:
df_cleaned = pd.DataFrame(unique_contracts)

print(df_cleaned["clean_context"].str.len().describe())

count       510.000000
mean      50592.280392
std       53909.954570
min         645.000000
25%       16134.500000
50%       32097.500000
75%       65329.750000
max      300304.000000
Name: clean_context, dtype: float64


## chunking (taille fixe + overlap)

In [54]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document
from collections import defaultdict

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

Splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    tokenizer=tokenizer.encode
)

documents = [
    Document(
        text=c["clean_context"], metadata={"title": c["title"]})
        for c in unique_contracts
]

nodes = Splitter.get_nodes_from_documents(documents)

# chunk index for each contract
counters = defaultdict(int)
for n in nodes:
    title = n.metadata["title"]
    n.metadata["chunk_index"] = counters[title]
    counters[title] += 1

print(f"Number of contracts : {len(documents)}")
print(f"Number of chunks: {len(nodes)}")
print(f"Average chunks per contract: {len(nodes)/len(documents):.1f}")

Number of contracts : 510
Number of chunks: 14230
Average chunks per contract: 27.9


In [55]:
chunks_per_contract = pd.Series(counters)
print(chunks_per_contract.describe())

count    510.000000
mean      27.901961
std       30.699734
min        1.000000
25%        9.000000
50%       17.000000
75%       35.000000
max      214.000000
dtype: float64


In [56]:
chunk_token_lengths = [len(tokenizer.encode(n.get_content())) for n in nodes]
pd.Series(chunk_token_lengths).describe()

count    14230.000000
mean       408.228461
std         63.663006
min          3.000000
25%        394.000000
50%        428.000000
75%        447.000000
max        497.000000
dtype: float64

In [57]:
sample_title = unique_contracts[0]["title"]
sample_nodes = [n for n in nodes if n.metadata["title"]==sample_title]

print(f"contract title: {sample_title}")
print(f"total chunks for this contract: {len(sample_nodes)}\n")

for n in sample_nodes[:5]:
    print(f"--- chunk_index={n.metadata['chunk_index']} ({len(tokenizer.encode(n.get_content()))} tokens)")
    print(n.get_content())
    print()

contract title: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
total chunks for this contract: 18

--- chunk_index=0 (453 tokens)
EXHIBIT 10.6

DISTRIBUTOR AGREEMENT

THIS DISTRIBUTOR AGREEMENT (the "Agreement") is made by and between Electric City Corp., a Delaware corporation ("Company") and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

RECITALS

A. The Company's Business. The Company is presently engaged in the business of selling an energy efficiency device, which is referred to as an "Energy Saver" which may be improved or otherwise changed from its present composition (the "Products"). The Company may engage in the business of selling other products or other devices other than the Products, which will be considered Products if Distributor exercises its options pursuant to Section 7 hereof.

B. Representations. As an inducement to the Company to enter into this Agreement, the Distributor has represented that it has or will have the facilities

In [58]:
contracts_by_title = {c["title"]: c for c in unique_contracts}

single_chunk_titles = [title for title, count in counters.items() if count==1]
print(f"contracts with only 1 chunk : {len(single_chunk_titles)} / {len(unique_contracts)}\n")

for title in single_chunk_titles:
    text = contracts_by_title[title]["clean_context"]
    n_tokens = len(tokenizer.encode(text))
    print(f"{title[:70]:<70} | {len(text):>6} chars | {n_tokens:>4} tokens")

contracts with only 1 chunk : 16 / 510

NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT                       |   1081 chars |  226 tokens
GALACTICOMMTECHNOLOGIESINC_11_07_1997-EX-10.46-WEB HOSTING AGREEMENT   |   1454 chars |  372 tokens
VIRGINGALACTICHOLDINGS,INC_04_08_2020-EX-99.1-JOINT FILING STATEMENT   |   1194 chars |  246 tokens
BANGIINC_05_25_2005-EX-10-Premium Managed Hosting Agreement            |   1973 chars |  446 tokens
BLACKROCKMUNIHOLDINGSINVESTMENTQUALITYFUND_04_07_2020-EX-99.01-JOINT F |   1283 chars |  271 tokens
SPRINGBANKPHARMACEUTICALS,INC_04_08_2020-EX-99.A-JOINT FILING AGREEMEN |   1556 chars |  370 tokens
UNITEDNATIONALBANCORP_03_03_1999-EX-99-Outsourcing Agreement with the  |   1867 chars |  378 tokens
LejuHoldingsLtd_20140121_DRS (on F-1)_EX-10.26_8473102_EX-10.26_Conten |   2112 chars |  451 tokens
MACY_S,INC_05_11_2020-EX-99.4-JOINT FILING AGREEMENT                   |   1397 chars |  329 tokens
GluMobileInc_20070319_S-1A_EX-10.09_436630_EX-10.09_Content 

## Embedding

In [59]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import time

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

sample_chunk = nodes[0].get_content()
start = time.time()
vec = embed_model.get_text_embedding(sample_chunk)
elapsed = time.time() - start

print(f"Embedding dimension: {len(vec)}")
print(f"time for 1 chunk: {elapsed:.3f}s")
print(f"first 5 values: {vec[:5]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10105.04it/s]


Embedding dimension: 384
time for 1 chunk: 1.108s
first 5 values: [-0.019716884940862656, -0.013978542760014534, 0.03723134845495224, -0.017645804211497307, 0.0216973889619112]


In [60]:
sample_nodes = nodes[:50]

start = time.time()
sample_vecs = embed_model.get_text_embedding_batch([n.get_content() for n in sample_nodes])
elapsed = time.time() - start

print(f"time for {len(sample_nodes)} chunks: {elapsed:.2f}s ({elapsed/len(sample_nodes)*1000:.1f} ms/chunk)")

estimated_total = elapsed/len(sample_nodes)*len(nodes)
print(f"estimated time for all {len(nodes)} chunks: {estimated_total/60:.1f} minutes")

time for 50 chunks: 1.83s (36.5 ms/chunk)
estimated time for all 14230 chunks: 8.7 minutes


## indexation

In [61]:
from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore

client = QdrantClient(path="../artifacts/qdrant_storage")
vector_store = QdrantVectorStore(client=client, collection_name="cuad_contracts")

In [62]:
from llama_index.core import VectorStoreIndex, StorageContext

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    embed_model=embed_model,
    storage_context=storage_context,
    show_progress=True,
)

Generating embeddings: 100%|██████████| 2048/2048 [00:25<00:00, 78.85it/s]
/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/llama_index/vector_stores/qdrant/base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(
Generating embeddings: 100%|██████████| 1942/1942 [00:23<00:00, 83.11it/s]


In [64]:
retriever = index.as_retriever(similarity_top_k=3)
results = retriever.retrieve("what is the cap on liability?")

for r in results:
    print(f"score={r.score:.3f} | title={r.metadata['title'][:60]} | chunk_index={r.metadata['chunk_index']}")
    print(r.get_content())
    print()

score=0.751 | title=REGANHOLDINGCORP_03_31_2008-EX-10-LICENSE AND HOSTING AGREEM | chunk_index=14
TAG shall inform LMG of any such Claim against TAG and shall have the right to participate in the defense of any such suit or proceeding at its expense and through counsel of its choosing.

Limitation of Liability. Neither party shall be liable to the other pursuant to this Agreement for any amounts representing loss of profit, loss of business or indirect, consequential, exemplary, or punitive damages of the other party. The foregoing shall not limit the indemnification, defense and hold harmless obligations set forth in this Agreement other than those set forth in Section 5.4 and shall not apply with respect to damages or losses arising from the wrongful termination of this Agreement by LMG, willful misconduct, gross negligence or breach of LMG's obligations under Section 3. 8.

score=0.740 | title=AtnInternationalInc_20191108_10-Q_EX-10.1_11878541_EX-10.1_M | chunk_index=12
2.24 Liabili

In [65]:
results = retriever.retrieve("what is the governing law of this agreement?")

for r in results:
    print(f"score={r.score:.3f} | title={r.metadata['title'][:60]} | chunh_index={r.metadata['chunk_index']}")
    print(r.get_content())
    print()

score=0.763 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMEN | chunh_index=5
Governing Law. This Agreement shall be governed by and construed in accordance with the laws of the Province of British Columbia (without regard to its conflict of laws provisions) which shall be deemed to be the proper law thereof. All disputes or claims arising out of or in relation to the Agreement may be submitted to and resolved by the Courts of the Province of British Columbia (including the Supreme Court of Canada). The parties hereby irrevocably submit and attorn to the non-exclusive jurisdiction of such Courts to finally adjudicate or determine any suit, action, or proceeding arising out of or in relation to this Agreement.

3

13. Terms Paramount. The terms of this Agreement are in addition to any other existing terms (the "Existing Terms") between the Company and Turpin. In

the event of any inconsistency or conflict between the terms of this Agreement and any Existing Terms, the

## Generation

In [74]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="llama3.2:3b", request_timeout=120, temperature=0.0)

response = llm.complete("say hello in one sentence.")
print(response)

Hello!


In [76]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(retriever=retriever,
                                              llm=llm)

response = query_engine.query("what is the governing law of this agreement?")
print(response)
print("\n--- source chunks used ---")
for node in response.source_nodes:
    print(f"score={node.score:.3f} | title={node.metadata['title']} | chunk_index={node.metadata['chunk_index']}")

The governing law for this agreement is the laws of the Province of British Columbia (without regard to its conflict of law provisions) in some cases, but for other agreements, it is the substantive laws of the State of New York.

--- source chunks used ---
score=0.763 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=5
score=0.760 | title=VERICELCORP_08_06_2019-EX-10.10-SUPPLY AGREEMENT | chunk_index=50
score=0.760 | title=INTERSECTENT,INC_05_11_2020-EX-10.1-SUPPLY AGREEMENT | chunk_index=17


In [78]:
questions = [
    "Is there a non-compete clause in this agreement?",
    "What happens if either party wants to terminate this agreement for convenience?",
    "What is th cap on liability for damages under this agreement?",
    "Does this agreement give either party audit rights"
]

for q in questions:
    response = query_engine.query(q)
    print(f"QUESTION: {q}")
    print(f"ANSWER: {response}\n")
    print("sources:")
    for node in response.source_nodes:
        print(f" score={node.score:.3f} | title={node.metadata['title']} | chunk_index={node.metadata['chunk_index']}")

    print("\n" + "="*100 + "\n")

QUESTION: Is there a non-compete clause in this agreement?
ANSWER: Yes, there are non-compete clauses in this agreement.

sources:
 score=0.792 | title=Quaker Chemical Corporation - NON COMPETITION AND NON SOLICITATION AGREEMENT | chunk_index=4
 score=0.770 | title=VIVINT SOLAR, INC. - NON-COMPETITION AGREEMENT | chunk_index=1
 score=0.765 | title=VIVINT SOLAR, INC. - NON-COMPETITION AGREEMENT | chunk_index=0


QUESTION: What happens if either party wants to terminate this agreement for convenience?
ANSWER: Neither party can terminate this agreement for convenience.

sources:
 score=0.781 | title=LEJUHOLDINGSLTD_03_12_2014-EX-10.34-INTERNET CHANNEL COOPERATION AGREEMENT | chunk_index=15
 score=0.777 | title=CHEETAHMOBILEINC_04_22_2014-EX-10.43-Cooperation Agreement | chunk_index=11
 score=0.771 | title=TUNIUCORP_03_06_2014-EX-10-COOPERATION AGREEMENT | chunk_index=4


QUESTION: What is th cap on liability for damages under this agreement?
ANSWER: The aggregate of all amounts paid under

In [81]:
sample_title = unique_contracts[107]["title"]

contract_qas = [ex for ex in eval_examples if ex['title'] == sample_title]

print(f"Contract: {sample_title}")
print(f"Number of questions for this contract: {len(contract_qas)}\n")

for qa in contract_qas[:20]:
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answers'] if qa['answers'] else 'no answer/clause absent'}")
    print()

Contract: SLOVAKWIRELESSFINANCECOBV_03_28_2001-EX-4.(B)(II).3-Maintenance and support contract for SICAP(R) modules
Number of questions for this contract: 42

Q: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
A: ['Maintenance and support contract']

Q: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
A: ['Sicap Ltd']

Q: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
A: ['EuroTel']

Q: Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
A: no answer/clause absent

Q: Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. D

In [86]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

target_title = "KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT"

filtred_retriever = index.as_retriever(
    similarity_top_k=3,
    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=target_title)]),

)

filtred_query_engine = RetrieverQueryEngine.from_args(retriever=filtred_retriever, llm=llm)

response = filtred_query_engine.query("what is the governing law of this agreement?")
print(response)
print("\nsources:")
for node in response.source_nodes:
    print(f" score={node.score:.3f} | title={node.metadata['title']} | chunk_index={node.metadata['chunk_index']}")

The laws of the Province of British Columbia govern this Agreement.

sources:
 score=0.763 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=5
 score=0.684 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=6
 score=0.670 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=4


## RAG Evaluation

In [87]:
from collections import defaultdict

grouped = defaultdict(list)
for ex in eval_examples:
    key = (ex['title'], ex['question'])
    grouped[key].extend(ex['answers'])

eval_dataset_raw = [
    {"title": title, "question": question, "ground_truth": answers}
    for (title, question), answers in grouped.items() if len(answers)>0
]

print(f"Number of (contract, question) pairs with at least one answer: {len(eval_dataset_raw)}")

Number of (contract, question) pairs with at least one answer: 6702


In [88]:
sample_title = unique_contracts[107]["title"]

for item in eval_dataset_raw:
    if item["title"] == sample_title and "Parties" in item["question"]:
        print(item["question"])
        print(item["ground_truth"])

Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
['Sicap Ltd', 'EuroTel']


In [89]:
import random
import time

def query_scoped(title, question):
    filtered_retriiever = index.as_retriever(
        similarity_top_k=3,
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )

    filtered_query_engine = RetrieverQueryEngine.from_args(retriever=filtered_retriiever, llm=llm)
    response = filtered_query_engine.query(question)
    contexts = [node.get_content() for node in response.source_nodes]
    return str(response), contexts

random.seed(42)
sample = random.sample(eval_dataset_raw, 5)

start = time.time()
for item in sample:
    answer, contexts = query_scoped(item['title'], item['question'])
    print(f"Contract: {item['title']}")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: {item['ground_truth']}")
    print()

elapsed = time.time() - start

print(f"time for {len(sample)} queries is {elapsed:.1f}s ({elapsed/len(sample):.1f}s/query)")
estimated_total_hours = elapsed/len(sample)*len(eval_dataset_raw)/3600
print(f"estimated time for all {len(eval_dataset_raw)} pairs: {estimated_total_hours:.1f} hours")


Contract: SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
Question: Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector?
Generated answer: The contract contains provisions related to non-solicitation and non-compete, but they are not explicitly stated as "Non-Compete" agreements. However, clauses 4 and 5 of the agreement can be interpreted as restrictions on the ability of a party to compete with the counterparty.

Clause 4 states that the Marketing Affiliate shall not directly or indirectly solicit an existing business customer of Equidata during the term and condition of this Agreement, except for joint marketing purposes. This provision may be seen as restricting the Marketing Affiliate's ability to compete with Equidata in certain markets or ind

In [92]:
def get_category(question):
    if '"' in question:
        return question.split('"')[1]
    return None

for item in eval_dataset_raw:
    item["category"]= get_category(item['question'])

by_category = defaultdict(list)
for item in eval_dataset_raw:
    by_category[item['category']].append(item)

rows= []
for category, items in by_category.items():
    lengths = [len(" ".join(it["ground_truth"])) for it in items]
    rows.append({
        "category": category,
        "n_examples": len(items),
        "avg_length_chars": sum(lengths)/len(lengths),
        "min_length_chars": min(lengths),
        "max_length_chars": max(lengths)
    })

summary_df = pd.DataFrame(rows).sort_values("avg_length_chars")
print(summary_df.to_string(index=False))

                          category  n_examples  avg_length_chars  min_length_chars  max_length_chars
                    Agreement Date         470         18.442553                 3               121
                     Document Name         510         30.482353                11               285
                    Effective Date         390         71.394872                 3               717
                           Parties         509        122.280943                17              1448
                     Governing Law         437        219.004577                42               885
                   Expiration Date         413        260.990315                12              1777
           Third Party Beneficiary          32        285.406250                64              1030
       Termination For Convenience         183        298.672131                35              2063
Notice Period To Terminate Renewal         111        337.279279               133         

In [93]:
random.seed(42)
for category, items in by_category.items():
    print(f"\n{'='*50}\nCATEGORY: {category} ({len(items)} examples)\n{'='*50}")
    sample_size = min(3, len(items))
    for item in random.sample(items, sample_size):
        print(f"\nContract: {item['title']}")
        print(f"Ground truth: {item['ground_truth']}")


CATEGORY: Document Name (510 examples)

Contract: EmbarkComInc_19991008_S-1A_EX-10.10_6487661_EX-10.10_Co-Branding Agreement
Ground truth: ['CO-BRANDING AGREEMENT']

Contract: OASYSMOBILE,INC_07_05_2001-EX-10.17-OUTSOURCING AGREEMENT
Ground truth: ['ASP and Outsourcing Agreement', 'Order Form']

Contract: IntegrityFunds_20200121_485BPOS_EX-99.E UNDR CONTR_11948727_EX-99.E UNDR CONTR_Service Agreement
Ground truth: ['DISTRIBUTION AND SERVICES AGREEMENT']

CATEGORY: Parties (509 examples)

Contract: IMAGEWARESYSTEMSINC_12_20_1999-EX-10.22-MAINTENANCE AGREEMENT
Ground truth: ['SAGEM S.A.', 'XIMAGE corporation', 'MORPHO', 'XIMAGE']

Contract: EmmisCommunicationsCorp_20191125_8-K_EX-10.6_11906433_EX-10.6_Marketing Agreement
Ground truth: ['MediaCo Holding Inc.', 'WBLS-WLIB LLC', 'Programmer', 'Licensee']

Contract: IGENEBIOTECHNOLOGYINC_05_13_2003-EX-1-JOINT VENTURE AGREEMENT
Ground truth: ['"PARTY" shall mean each of T&L and Igene', '"Igene"; collectively with T&L, the "Parties"', 'T&L', 

In [95]:
from llama_index.core import PromptTemplate

qa_prompt_template = PromptTemplate(
    "Context information from the contract is bellow.\n"
    "------------------\n"
    "{context_str}\n"
    "------------------\n"
    "You are a contract review assistant. Using ONLY the contract above, answer the question"
    "by quoting the exact relevant sentence(s) from the contract text as directly as possible."
    "Do not paraphrase, summarize, or add commentary or recommandations.\n"
    "If the context does not contain information relevant to the question, respond exactly with: "
    "\"This clause is not present in this contract.\"\n"
    "Question: {query_str}\n"
    "Answer: "
)

In [97]:
query_engine_v2 = RetrieverQueryEngine.from_args(
    retriever=retriever,
    llm=llm,
    text_qa_template=qa_prompt_template

)

random.seed(42)
sample = random.sample(eval_dataset_raw, 5)

for item in sample:
    filtered_retriever = index.as_retriever(
        similarity_top_k=3,
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=item["title"])])

    )
    query_engine_v2 = RetrieverQueryEngine.from_args(
        retriever=filtered_retriever, llm=llm, text_qa_template=qa_prompt_template
    )

    response = query_engine_v2.query(item['question'])
    print(f"Contract: {item['title']}")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {response}")
    print(f"Ground truth: {item['ground_truth']}")
    print()

Contract: SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
Question: Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector?
Generated answer: 4. Non-solicitation of Clients. Marketing Affiliate shall not directly or indirectly solicit an existing business customer of Equidata during the term and condition of this Agreement other than for joint marketing purposes.

5. Compliance. Marketing Affiliate nor Equidata, shall engage in any practice or activity that is not in compliance with the Fair Credit Reporting Act (FCRA), the Fair Debt Collection Practices Act (FDCPA) and the Health Insurance Portability and Accountability Act (HIPAA) as well as, but not limited to, any practice or activity that:

5.1. Violates any applicable law or regulation; including

In [98]:
category_to_tier = {}

for _, row in summary_df.iterrows():
    if row["avg_length_chars"] < 150:
        tier = 1
    elif row['avg_length_chars'] < 900:
        tier = 2
    else:
        tier = 3
    
    category_to_tier[row['category']] = tier

print(category_to_tier)

{'Agreement Date': 1, 'Document Name': 1, 'Effective Date': 1, 'Parties': 1, 'Governing Law': 2, 'Expiration Date': 2, 'Third Party Beneficiary': 2, 'Termination For Convenience': 2, 'Notice Period To Terminate Renewal': 2, 'Renewal Term': 2, 'Anti-Assignment': 2, 'No-Solicit Of Customers': 2, 'Volume Restriction': 2, 'No-Solicit Of Employees': 2, 'Most Favored Nation': 2, 'Price Restrictions': 2, 'Uncapped Liability': 2, 'Covenant Not To Sue': 2, 'Non-Disparagement': 2, 'Unlimited/All-You-Can-Eat-License': 2, 'Liquidated Damages': 2, 'Warranty Duration': 2, 'Competitive Restriction Exception': 2, 'Change Of Control': 2, 'Exclusivity': 2, 'Joint Ip Ownership': 2, 'Non-Compete': 2, 'Revenue/Profit Sharing': 2, 'Minimum Commitment': 3, 'Non-Transferable License': 3, 'Cap On Liability': 3, 'Audit Rights': 3, 'Ip Ownership Assignment': 3, 'Insurance': 3, 'Affiliate License-Licensee': 3, 'Post-Termination Services': 3, 'License Grant': 3, 'Irrevocable Or Perpetual License': 3, 'Rofr/Rofo/Ro

In [100]:
def retrieve_scoped(title, question, category):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata['title']==title]

    if tier == 1:
        contract_nodes_sorted = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes_sorted[:2]
        return [n.get_content() for n in selected]
    
    top_k=3 if tier == 2 else 10
    filtered_retriever = index.as_retriever(
        similarity_top_k=top_k,
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)
    return [n.get_content() for n in retrieved_nodes]

In [101]:
def generate_answer(title, question, category):
    contexts = retrieve_scoped(title, question, category)
    context_str = "\n\n".join(contexts)
    prompt = qa_prompt_template.format(context_str=context_str, query_str=question)
    response = llm.complete(prompt)
    return str(response), contexts


In [104]:
random.seed(42)
sample = random.sample(eval_dataset_raw, 5)

for item in sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: {item['ground_truth']}")
    print()

Contract: SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
Category: Non-Compete (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector?
Generated answer: 4. Non-solicitation of Clients. Marketing Affiliate shall not directly or indirectly solicit an existing business customer of Equidata during the term and condition of this Agreement other than for joint marketing purposes.

5. Compliance. Marketing Affiliate nor Equidata, shall engage in any practice or activity that is not in compliance with the Fair Credit Reporting Act (FCRA), the Fair Debt Collection Practices Act (FDCPA) and the Health Insurance Portability and Accountability Act (HIPAA) as well as, but not limited to, any practice or activity that:

5.1. Violates any applicab

In [105]:
qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------\n"
    "{context_str}\n"
    "---------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1.decide wheter the context above contains information relevant to the question."
    "a relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names"
    "- not only comlete grammatical sentences.\n"
    "2.if relevant information exists respond with ONLY the exact text from the context that answers"
    "the question. quote it directly. don not paraphrase, do not add explanation or recommandations.\n"
    "3.if nothing in the context is relevant to the question, respond with exactly this sentence and"
    "nothing else: \"This clause is not present in this contract.\"\n"
    "never combine both - give exactly one of the two response types above, never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)

In [122]:
def  retrieve_scoped(title, question, category):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier ==1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [n.get_content() for n in selected]
    
    filtered_retriever = index.as_retriever(
        similarity_top_k=3,
        filters= MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)
    return [n.get_content() for n in retrieved_nodes]

In [108]:
import random
random.seed(1)
sample_titles = random.sample([c['title'] for c in unique_contracts], 10)

for title in sample_titles:
    first_chunks = [n for n in nodes if n.metadata["title"]== title and n.metadata["chunk_index"] in (0, 1)]
    first_chunks_sorted = sorted(first_chunks, key=lambda n: n.metadata["chunk_index"])
    print(f"{'='*100}\n{title}\n{'='*100}")
    for n in first_chunks_sorted:
        print(f"------ chunk_index={n.metadata['chunk_index']} ------")
        print(n.get_content())
        print()


INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRANCHISE AGREEMENT
------ chunk_index=0 ------
FRANCHISE AGREEMENT

TABLE OF CONTENTS

I. INTRODUCTION

II. AGREEMENT

1. Definitions

2. Franchise Grant; Term

2.1 Grant 2.2 Term 2.3 No Renewal Right: No Exclusivity 2.4 Continuous Operation 2.5 Best Efforts

3. Consideration for Franchise Grant

4. Management, Control and Corporate Documents of Franchisee

4.1 Managing Director 4.2 Director of Operations 4.3 Substitute Director of Operations 4.4 Restaurant Manager 4.5 Corporate Documents 4.5.1 Single Purpose Entity 4.5 2 Managing Director's Authority 4.5.3 Issuance and Transfer of Shares 4.5.4 Amendments

5. Standards and Uniformity 5.1 Strict Compliance 5.2 The MOD Manual 5.3 Building and Premises 5.3.1 Initial Construction 5 3 2 Repair and Maintenance 5 3.3 Current Image 5.4 Signs 5.5 Equipment 5.6 Vending Machines, Etc.

5 7 Menu Service and Hygiene 5.8 Hours of Operations 5.9 Uniforms 5.10 Advertising and Promotion Materials 5.11 Interfe

In [118]:
for item in eval_dataset_raw:

    if item["title"] == "StaarSurgicalCompany_20180801_10-Q_EX-10.37_11289449_EX-10.37_Distributor Agreement" and item["question"] == 'Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. Details: The date when the contract is effective':
        res = item["ground_truth"]
        print(res)

['____________']


In [119]:
category_to_tier["Agreement Date"] = 2
category_to_tier["Effective Date"] = 2

print(category_to_tier)

{'Agreement Date': 2, 'Document Name': 1, 'Effective Date': 2, 'Parties': 1, 'Governing Law': 2, 'Expiration Date': 2, 'Third Party Beneficiary': 2, 'Termination For Convenience': 2, 'Notice Period To Terminate Renewal': 2, 'Renewal Term': 2, 'Anti-Assignment': 2, 'No-Solicit Of Customers': 2, 'Volume Restriction': 2, 'No-Solicit Of Employees': 2, 'Most Favored Nation': 2, 'Price Restrictions': 2, 'Uncapped Liability': 2, 'Covenant Not To Sue': 2, 'Non-Disparagement': 2, 'Unlimited/All-You-Can-Eat-License': 2, 'Liquidated Damages': 2, 'Warranty Duration': 2, 'Competitive Restriction Exception': 2, 'Change Of Control': 2, 'Exclusivity': 2, 'Joint Ip Ownership': 2, 'Non-Compete': 2, 'Revenue/Profit Sharing': 2, 'Minimum Commitment': 3, 'Non-Transferable License': 3, 'Cap On Liability': 3, 'Audit Rights': 3, 'Ip Ownership Assignment': 3, 'Insurance': 3, 'Affiliate License-Licensee': 3, 'Post-Termination Services': 3, 'License Grant': 3, 'Irrevocable Or Perpetual License': 3, 'Rofr/Rofo/Ro

In [ ]:
qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "3. If relevant information exists, respond with ONLY the exact text from the context that answers "
    "the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "4. If nothing in the context is relevant to the question, respond with exactly this sentence and "
    "nothing else: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above, never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


In [ ]:
random.seed(42)
sample = random.sample(eval_dataset_raw, 5)

for item in sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: {item['ground_truth']}")
    print()


Contract: SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
Category: Non-Compete (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector?
Generated answer: 4. Non-solicitation of Clients. Marketing Affiliate shall not directly or indirectly solicit an existing business customer of Equidata during the term and condition of this Agreement other than for joint marketing purposes. Further, Marketing Affiliate shall not market similar products from competing companies on any Web Site Landing Page containing the Equidata or Marketing Affiliate Web link as long as this Agreement is in effect.

5. Compliance. Marketing Affiliate nor Equidata, shall engage in any practice or activity that is not in compliance with the Fair Credit Reporting Act 

In [124]:
qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "3. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "4. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "5. If nothing in the context is relevant to the question, respond with exactly this sentence and "
    "nothing else: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above, never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


In [125]:
random.seed(42)
sample = random.sample(eval_dataset_raw, 5)

for item in sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: {item['ground_truth']}")
    print()

Contract: SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
Category: Non-Compete (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector?
Generated answer: "NON-SOLICITATION OF CLIENTS"

5. Non-solicitation of Clients. Marketing Affiliate shall not directly or indirectly solicit an existing business customer of Equidata during the term and condition of this Agreement other than for joint marketing purposes. Further, Marketing Affiliate shall not market similar products from competing companies on any Web Site Landing Page containing the Equidata or Marketing Affiliate Web link as long as this Agreement is in effect.

Specifically, this clause restricts the Marketing Affiliate from operating in a certain geography (the entire United Sta

In [126]:
no_answer_dataset = [
    {"title": title, "question": question, "category": get_category(question)}
    for (title, question), answers in grouped.items() if len(answers)==0
]

print(f"Number of (contract, question) pairs with No answer (clause absent): {len(no_answer_dataset)}")

Number of (contract, question) pairs with No answer (clause absent): 14208


In [128]:
random.seed(42)
no_answer_sample = random.sample(no_answer_dataset, 5)

for item in no_answer_sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: clause absent of this contract")
    print()

Contract: OPTIMIZEDTRANSPORTATIONMANAGEMENT,INC_07_26_2000-EX-6.6-DISTRIBUTOR AGREEMENT
Category: Unlimited/All-You-Can-Eat-License (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Unlimited/All-You-Can-Eat-License" that should be reviewed by a lawyer. Details: Is there a clause granting one party an “enterprise,” “all you can eat” or unlimited usage license?
Generated answer: "UNLIMITED LICENSE NOT SPECIFIED IN THIS CONTRACT."
Ground truth: clause absent of this contract

Contract: ELECTRAMECCANICA VEHICLES CORP. - Manufacturing Agreement
Category: Source Code Escrow (Tier 3)
Question: Highlight the parts (if any) of this contract related to "Source Code Escrow" that should be reviewed by a lawyer. Details: Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Generated answer: 9.7 Exceptions
Ground truth: clause absent

In [129]:
qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. A section heading or title that merely resembles or matches the category name in the question is "
    "NOT sufficient evidence by itself. You must verify that the actual clause text beneath that heading "
    "substantively satisfies what the question's \"Details\" describes. If only a heading is found, with no "
    "clause content confirming the details, treat this as not relevant — a heading alone is not an answer.\n"
    "3. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "4. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "5. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "6. If nothing in the context is relevant to the question (including cases covered by rule 2), respond "
    "with EXACTLY the following sentence, word for word, with no rephrasing or additions: "
    "\"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above (step 5 or step 6), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


In [130]:
random.seed(42)
no_answer_sample = random.sample(no_answer_dataset, 5)

for item in no_answer_sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: clause absent of this contract")
    print()

Contract: OPTIMIZEDTRANSPORTATIONMANAGEMENT,INC_07_26_2000-EX-6.6-DISTRIBUTOR AGREEMENT
Category: Unlimited/All-You-Can-Eat-License (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Unlimited/All-You-Can-Eat-License" that should be reviewed by a lawyer. Details: Is there a clause granting one party an “enterprise,” “all you can eat” or unlimited usage license?
Generated answer: This clause is not present in this contract.
Ground truth: clause absent of this contract

Contract: ELECTRAMECCANICA VEHICLES CORP. - Manufacturing Agreement
Category: Source Code Escrow (Tier 3)
Question: Highlight the parts (if any) of this contract related to "Source Code Escrow" that should be reviewed by a lawyer. Details: Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Generated answer: 9.7 Exceptions
Ground truth: clause absent of thi

In [132]:
random.seed(42)
sample = random.sample(eval_dataset_raw, 5)

for item in sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: {item['ground_truth']}")
    print()

Contract: SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
Category: Non-Compete (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector?
Generated answer: This clause is not present in this contract.
Ground truth: ['Further, Marketing  Affiliate shall not market similar products from competing companies on any Web Site Landing Page containing the Equidata or Marketing Affiliate Web link as long as this Agreement is in effect.']

Contract: PACIFICSYSTEMSCONTROLTECHNOLOGYINC_08_24_2000-EX-10.53-SPONSORSHIP AGREEMENT
Category: Document Name (Tier 1)
Question: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
Generated answer: SPONSORSHIP AGREEME

In [133]:
qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. Do not treat a bare section heading or title (with no accompanying clause text) as an answer just "
    "because it resembles the category name in the question. However, if the context contains actual clause "
    "text — obligations, rights, restrictions, or definitions — that addresses what the question's \"Details\" "
    "describes, that IS valid evidence: quote it normally. Only treat something as \"not present\" when there "
    "is no clause text addressing the details, not merely because a heading happens to be present.\n"
    "3. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "4. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "5. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "6. If nothing in the context is relevant to the question, respond with EXACTLY the following sentence, "
    "word for word, with no rephrasing or additions: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above (step 5 or step 6), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


In [134]:
random.seed(42)
sample = random.sample(eval_dataset_raw, 5)

for item in sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: {item['ground_truth']}")
    print()

Contract: SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
Category: Non-Compete (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector?
Generated answer: 5.1. Violates any applicable law or regulation; including but not limited to the sale of illegal goods or the violation of export control or obscenity laws; that invade the privacy of any third party; that are in any way connected with the transmission of "junk mail", "spam" or the unsolicited mass distribution of e-mail, or with any unethical marketing practices.
Ground truth: ['Further, Marketing  Affiliate shall not market similar products from competing companies on any Web Site Landing Page containing the Equidata or Marketing Affiliate Web link as long as this Agreement is in 

In [135]:
random.seed(42)
no_answer_sample = random.sample(no_answer_dataset, 5)

for item in no_answer_sample:
    tier = category_to_tier.get(item["category"], 2)
    answer, contexts = generate_answer(item["title"], item["question"], item["category"])
    print(f"Contract: {item['title']}")
    print(f"Category: {item['category']} (Tier {tier})")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {answer}")
    print(f"Ground truth: clause absent of this contract")
    print()

Contract: OPTIMIZEDTRANSPORTATIONMANAGEMENT,INC_07_26_2000-EX-6.6-DISTRIBUTOR AGREEMENT
Category: Unlimited/All-You-Can-Eat-License (Tier 2)
Question: Highlight the parts (if any) of this contract related to "Unlimited/All-You-Can-Eat-License" that should be reviewed by a lawyer. Details: Is there a clause granting one party an “enterprise,” “all you can eat” or unlimited usage license?
Generated answer: This clause is not present in this contract.
Ground truth: clause absent of this contract

Contract: ELECTRAMECCANICA VEHICLES CORP. - Manufacturing Agreement
Category: Source Code Escrow (Tier 3)
Question: Highlight the parts (if any) of this contract related to "Source Code Escrow" that should be reviewed by a lawyer. Details: Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Generated answer: 9.7 Exceptions Each party agrees that the terms a

In [140]:
import random
random.seed(42)

N_POSITIVE_PER_CATEGORY = 5
N_NEGATIVE_PER_CATEGORY = 3

positive_sample = []
for category, items in by_category.items():
    n = min(N_POSITIVE_PER_CATEGORY, len(items))
    positive_sample.extend(random.sample(items, n))

print(f"positive sample size: {len(positive_sample)} (across {len(by_category)} categories)")

no_answer_by_category = defaultdict(list)
for item in no_answer_dataset:
    no_answer_by_category[item["category"]].append(item)

negative_sample = []
for category, items in no_answer_by_category.items():
    n = min(N_NEGATIVE_PER_CATEGORY, len(items))
    negative_sample.extend(random.sample(items, n))

print(f"negative sample size: {len(negative_sample)} (across {len(no_answer_by_category)} categories)")


positive sample size: 205 (across 41 categories)
negative sample size: 118 (across 40 categories)


In [141]:
for item in positive_sample:
    item["ground_truth_clean"] = [clean_contract_text(a) for a in item["ground_truth"]]

In [ ]:
llm = Ollama(model="llama3.2:3b", request_timeout=300, temperature=0.0)

In [147]:
from tqdm import tqdm
import json

def run_generation(sample, is_positive, output_path):
    results = []
    for item in tqdm(sample):
        try:
            answer, contexts = generate_answer(item["title"], item["question"], item["category"])
        except Exception as e:
            answer = f"[ERROR: {type(e).__name__}: {e}]"
            context = []

        result = {
            "title" : item["title"],
            "category" : item["category"],
            "tier" : category_to_tier.get(item["category"], 2),
            "question" : item["question"],
            "answer" : answer,
            "contexts" : contexts,
            "is_positive" : is_positive,
        }
    
        if is_positive:
            result["ground_truth"] = " ".join(item["ground_truth_clean"])
        else:
            result["ground_truth"] = "This clause in not present in this contract."
    
        results.append(result)
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2)
    return results


In [148]:
positive_results = run_generation(positive_sample, is_positive=True, output_path="../artifacts/positive_results_v1.json")
print(f"saved {len(positive_results)} positive results.")

100%|██████████| 205/205 [45:58<00:00, 13.46s/it]   

saved 205 positive results.


In [149]:
negative_results = run_generation(negative_sample, is_positive=False, output_path="../artifacts/negative_results_v1.json")
print(f"saved {len(negative_results)} negative results.")

100%|██████████| 118/118 [16:48<00:00,  8.55s/it]

saved 118 negative results.


In [152]:
import json
from collections import Counter

with open("../artifacts/negative_results_v1.json") as f:
    negative_results = json.load(f)

print(f"loaded {len(negative_results)} negative results")

ABSENCE_PHRASES = [
    "not present", "not specified", "not mentioned", "does not contain", 
    "no clause", "not applicable", "not included", "not found", "n/a", "absent"
]

def classify_negative(answer):
    if answer.startswith("[ERROR"):
        return "error"
    
    answer_lower = answer.lower()
    if any(phrase in answer_lower for phrase in ABSENCE_PHRASES):
        return "true negative"
    return "false positive"

for item in negative_results:
    item["classification"] = classify_negative(item["answer"])

counts = Counter(item["classification"] for item in negative_results)
print(counts)

total_classifiable = counts["true negative"] + counts["false positive"]
fp_rate = counts["false positive"] / total_classifiable * 100
print(f"False positive rate: {fp_rate:.1f}% ({counts['false positive']}/{total_classifiable})")

loaded 118 negative results
Counter({'false positive': 76, 'true negative': 42})
False positive rate: 64.4% (76/118)


In [155]:
import pandas as pd

df_neg = pd.DataFrame(negative_results)
df_neg_classifiable = df_neg[df_neg["classification"] != "error"]

fp_by_tier = df_neg_classifiable.groupby("tier")["classification"].apply(lambda x: (x == "false positive").mean() *100)
print(fp_by_tier)

tier
1    100.000000
2     65.384615
3     61.538462
Name: classification, dtype: float64


In [157]:
summary = df_neg_classifiable.groupby("category")["classification"].value_counts().unstack(fill_value=0)

for col in ["true negative", "false positive"]:
    if col not in summary.columns:
        summary[col] = 0

summary["total"] = summary["true negative"] + summary["false positive"]
summary["fp_rate_%"] = (summary["false positive"] / summary["total"] * 100).round(1)

summary = summary[["true negative", "false positive", "total", "fp_rate_%"]].sort_values("fp_rate_%", ascending=False)
print(summary.to_string())

classification                      true negative  false positive  total  fp_rate_%
category                                                                           
Affiliate License-Licensee                      0               3      3      100.0
Volume Restriction                              0               3      3      100.0
Renewal Term                                    0               3      3      100.0
Parties                                         0               1      1      100.0
Non-Disparagement                               0               3      3      100.0
Affiliate License-Licensor                      0               3      3      100.0
Liquidated Damages                              0               3      3      100.0
Joint Ip Ownership                              0               3      3      100.0
Irrevocable Or Perpetual License                0               3      3      100.0
Insurance                                       0               3      3    

In [3]:
import json
with open("../artifacts/positive_results_v1.json") as f:
    positive_results = json.load(f)

print(f"loaded {len(positive_results)} positive results")

positive_results_clean = [r for r in positive_results if not r["answer"].startswith("[ERROR")]
print(f"{len(positive_results_clean)} usable after removing errors ({len(positive_results) - len(positive_results_clean)} errors)")

from ragas import EvaluationDataset

ragas_data = [
    {
        "user_input": item["question"],
        "response": item["answer"],
        "retrieved_contexts": item["contexts"],
        "reference": item["ground_truth"],
    }
    for item in positive_results_clean
]

eval_dataset = EvaluationDataset.from_list(ragas_data)
print(eval_dataset)

loaded 205 positive results
202 usable after removing errors (3 errors)


/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=202)


In [11]:
from llama_index.llms.ollama import Ollama
llm = Ollama(model="llama3.2:3b", request_timeout=180, temperature=0.0)

In [12]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5860.15it/s]


In [13]:
from ragas.llms import LlamaIndexLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper

ragas_llm = LlamaIndexLLMWrapper(llm)
ragas_embeddings = LlamaIndexEmbeddingsWrapper(embed_model)

print("ragas llm:", ragas_llm)
print("ragas embeddings:", ragas_embeddings)

ragas llm: LlamaIndexLLMWrapper(llm=Ollama(...))
ragas embeddings: LlamaIndexEmbeddingsWrapper(embeddings=HuggingFaceEmbedding(...))


In [10]:
from ragas.metrics import faithfulness, answer_correctness, answer_relevancy, context_precision, context_recall
from ragas import evaluate
import random, time

metrics = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]

random.seed(42)
benchmark_data = random.sample(ragas_data, 10)
benchmark_datataset = EvaluationDataset.from_list(benchmark_data)

start = time.time()
benchmark_result = evaluate(
    dataset=benchmark_datataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

elapsed = time.time() - start
print(benchmark_result)
print(f"\ntimee for {len(benchmark_data)} examples: {elapsed:.1f}s ({elapsed/len(benchmark_data):.1f}s/example)")

estimated_total_minutes = elapsed / len(benchmark_data) * len(ragas_data) / 60
print(f"estimated time for all {len(ragas_data)} examples: {estimated_total_minutes:.1f} minutes")

Evaluating:   8%|▊         | 4/50 [01:45<14:42, 19.19s/it]Exception raised in Job[0]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Evaluating:  52%|█████▏    | 26/50 [05:41<05:32, 13.86s/it]Exception raised in Job[22]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[25]: TimeoutError()
Exception raised in Job[27]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Evaluating:  54%|█████▍    | 27/50 [06:00<05:45, 15.03s/it]Exception raised in Job[26]: TimeoutError()
Exception raised in Job[32]: TimeoutErr

{'faithfulness': 0.3750, 'answer_relevancy': 0.5517, 'context_precision': 1.0000, 'context_recall': 0.5387, 'answer_correctness': nan}

timee for 10 examples: 570.6s (57.1s/example)
estimated time for all 202 examples: 192.1 minutes


In [15]:
from ragas.run_config import RunConfig

run_config = RunConfig(timeout=180, max_workers=1)

random.seed(42)
benchmark_data = random.sample(ragas_data, 10)
benchmark_datataset = EvaluationDataset.from_list(benchmark_data)

start = time.time()
benchmark_result = evaluate(
    dataset=benchmark_datataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config
)
elapsed = time.time() - start
print(benchmark_result)
print(f"\ntimee for {len(benchmark_data)} examples: {elapsed:.1f}s ({elapsed/len(benchmark_data):.1f}s/example)")

estimated_total_minutes = elapsed / len(benchmark_data) * len(ragas_data) / 60
print(f"estimated time for all {len(ragas_data)} examples: {estimated_total_minutes:.1f} minutes")

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]Exception raised in Job[0]: OutputParserException(Invalid json output: Section 8.01. Any purported assignment without such consent shall be void.
Notwithstanding the foregoing, if any Party to this Agreement (or any of its successors or permitted assigns) (a) shall enter into a consolidation or merger transaction in which such Party is not the surviving entity and the surviving entity acquires or assumes all or substantially all of such Party's assets,
(b) shall transfer all or substantially all of such Party's assets to any Person or
(c) shall assign this Agreement to such Party's Affiliates, then, in each such case, the assigning Party (or its successors or permitted assigns, as applicable) shall ensure that the assignee or successor-in-interest expressly assumes in writing all of the obligations of the assigning Party under this Agreement,
and the assigning Party shall not be required to seek consent,
but shall provide written notice

{'faithfulness': 0.3750, 'answer_relevancy': 0.6002, 'context_precision': 0.9167, 'context_recall': 0.6259, 'answer_correctness': 0.5340}

timee for 10 examples: 1978.5s (197.9s/example)
estimated time for all 202 examples: 666.1 minutes


In [27]:
from dotenv import load_dotenv
load_dotenv()

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper

class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            return
        super().__setattr__(name, value)

claude_llm = ChatAnthropicNoTemp(model="claude-sonnet-5", max_tokens=4096)
ragas_llm_claude = LangchainLLMWrapper(claude_llm)

In [28]:
from ragas.run_config import RunConfig
import random, time

run_config_claude = RunConfig(timeout=180, max_workers=4)

random.seed(42)
benchmark_data = random.sample(ragas_data, 10)
benchmark_dataset = EvaluationDataset.from_list(benchmark_data)

start = time.time()
benchmark_result = evaluate(
    dataset = benchmark_dataset,
    metrics=metrics,
    llm=ragas_llm_claude,
    embeddings=ragas_embeddings,
    run_config=run_config_claude
)
elapsed = time.time() - start

print(benchmark_result)
print(f"\ntime for {len(benchmark_data)} examples : {elapsed:.1f}s ({elapsed/len(benchmark_data):.1f}s/example)")

Evaluating: 100%|██████████| 50/50 [02:29<00:00,  3.00s/it]

{'faithfulness': 0.8812, 'answer_relevancy': 0.6947, 'context_precision': 0.6000, 'context_recall': 0.5552, 'answer_correctness': 0.4791}

time for 10 examples : 150.1s (15.0s/example)


In [29]:
from dotenv import load_dotenv
load_dotenv()

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper

class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            return
        super().__setattr__(name, value)

claude_llm = ChatAnthropicNoTemp(model="claude-haiku-4-5", max_tokens=4096)
ragas_llm_claude = LangchainLLMWrapper(claude_llm)

In [30]:
from ragas.run_config import RunConfig
import random, time

run_config_claude = RunConfig(timeout=180, max_workers=4)

random.seed(42)
benchmark_data = random.sample(ragas_data, 10)
benchmark_dataset = EvaluationDataset.from_list(benchmark_data)

start = time.time()
benchmark_result = evaluate(
    dataset = benchmark_dataset,
    metrics=metrics,
    llm=ragas_llm_claude,
    embeddings=ragas_embeddings,
    run_config=run_config_claude
)
elapsed = time.time() - start

print(benchmark_result)
print(f"\ntime for {len(benchmark_data)} examples : {elapsed:.1f}s ({elapsed/len(benchmark_data):.1f}s/example)")

Evaluating: 100%|██████████| 50/50 [01:29<00:00,  1.80s/it]

{'faithfulness': 0.8142, 'answer_relevancy': 0.6930, 'context_precision': 0.6000, 'context_recall': 0.5686, 'answer_correctness': 0.5322}

time for 10 examples : 90.3s (9.0s/example)


In [32]:
import json
import time

BATCH_SIZE = 20
output_path = "../artifacts/ragas_scores_llama_haiku.json"

all_scores = []
start_total = time.time()

n_batches = (len(ragas_data) - 1) // BATCH_SIZE + 1

for i in range(0, len(ragas_data), BATCH_SIZE):
    batch = ragas_data[i:i + BATCH_SIZE]
    batch_dataset = EvaluationDataset.from_list(batch)

    print(f"batch {i // BATCH_SIZE + 1}/{n_batches} ({len(batch)} examples)...")

    result = evaluate(
        dataset=batch_dataset,
        metrics=metrics,
        llm=ragas_llm_claude,
        embeddings=ragas_embeddings,
        run_config=run_config_claude
    )

    batch_scores = result.to_pandas().to_dict(orient="records")
    all_scores.extend(batch_scores)

    with open(output_path, "w") as f:
        json.dump(all_scores, f, indent=2)

    print(f" -> saved ({len(all_scores)}/{len(ragas_data)} in total)")

elapsed_total = time.time() - start_total
print(f"finished in {elapsed_total/60:.1f} minutes")

batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:40<00:00,  1.00s/it]


 -> saved (20/202 in total)
batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [02:30<00:00,  1.51s/it]


 -> saved (40/202 in total)
batch 3/11 (20 examples)...


...

 -> saved (180/202 in total)
batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [03:16<00:00,  1.97s/it]


 -> saved (200/202 in total)
batch 11/11 (2 examples)...


Evaluating: 100%|██████████| 10/10 [00:18<00:00,  1.87s/it]

 -> saved (202/202 in total)
finished in 27.6 minutes


In [34]:
import json
import pandas as pd

with open("../artifacts/ragas_scores_llama_haiku.json") as f:
    scores_data = json.load(f)

with open("../artifacts/positive_results_v1.json") as f:
    positive_results = json.load(f)

positive_results_clean = [r for r in positive_results if not r["answer"].startswith("[ERROR")]

assert len(scores_data) == len(positive_results_clean)

for score_row, meta_row in zip(scores_data, positive_results_clean):
    score_row["category"] = meta_row["category"]
    score_row["tier"] = meta_row["tier"]
    score_row["title"] = meta_row["title"]

df_scores = pd.DataFrame(scores_data)

metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]

summary_global = df_scores[metric_cols].mean().to_frame(name="moyenne")
print(summary_global)

                     moyenne
faithfulness        0.757111
answer_relevancy    0.614114
context_precision   0.409241
context_recall      0.477275
answer_correctness  0.410722


In [35]:
summary_by_category = df_scores.groupby("category")[metric_cols].mean()
summary_by_category["n_examples"] = df_scores.groupby("category").size()
summary_by_category["score_moyen"] = summary_by_category[metric_cols].mean(axis=1)

summary_by_category_sorted = summary_by_category.sort_values("score_moyen")
print(summary_by_category_sorted.to_string())

                                    faithfulness  answer_relevancy  context_precision  context_recall  answer_correctness  n_examples  score_moyen
category                                                                                                                                          
Volume Restriction                      0.693681          0.479972           0.000000        0.100000            0.248242           4     0.304379
Covenant Not To Sue                     0.863636          0.522260           0.000000        0.000000            0.214668           5     0.320113
Most Favored Nation                     0.173333          0.727634           0.133333        0.400000            0.195733           5     0.326007
Agreement Date                          0.305556          0.609514           0.200000        0.200000            0.321533           5     0.327321
Rofr/Rofo/Rofn                          0.450000          0.729409           0.100000        0.200000            0.161

In [36]:
summary_global.to_csv("../artifacts/ragas_scores_llama_haiku_summary_global.csv")
summary_by_category_sorted.to_csv("../artifacts/ragas_scores_llama_haiku_summary_by_category.csv")

In [37]:
summary_by_category["retrieval_score"] = summary_by_category[["context_precision", "context_recall"]].mean(axis=1)
summary_by_category["generation_score"] = summary_by_category[["faithfulness", "answer_relevancy"]].mean(axis=1)

THRESHOLD = 0.5

def diagnose(row):
    retrieval_ok = row["retrieval_score"] >= THRESHOLD
    generation_ok = row["generation_score"] >= THRESHOLD
    if retrieval_ok and generation_ok:
        return "ok"
    elif not retrieval_ok and generation_ok:
        return "Retrieval issue"
    elif retrieval_ok and not generation_ok:
        return "Generation issue"
    else:
        return "Both retrieval and generation issues"
    

summary_by_category["diagnostic"] = summary_by_category.apply(diagnose, axis=1)

print(summary_by_category["diagnostic"].value_counts())
print()
print(summary_by_category[["retrieval_score", "generation_score", "diagnostic", "score_moyen"]].sort_values("score_moyen").to_string())

diagnostic
Retrieval issue                         23
ok                                      15
Both retrieval and generation issues     3
Name: count, dtype: int64

                                    retrieval_score  generation_score                            diagnostic  score_moyen
category                                                                                                                
Volume Restriction                         0.050000          0.586827                       Retrieval issue     0.304379
Covenant Not To Sue                        0.000000          0.692948                       Retrieval issue     0.320113
Most Favored Nation                        0.266667          0.450483  Both retrieval and generation issues     0.326007
Agreement Date                             0.200000          0.457535  Both retrieval and generation issues     0.327321
Rofr/Rofo/Rofn                             0.150000          0.589704                       Retrieval issue